In [1]:
import pandas as pd

df = pd.read_csv("HospitalOPD-noname.csv")

df_clean = df.drop(
    columns=["Unnamed: 2"],
    errors="ignore"
).copy()

timestamp_columns = [
    "prescription_start_datetime",
    "prescription_end_datetime",
    "lab_start_datetime",
    "lab_end_datetime",
    "imaging_start_datetime",
    "imaging_end_datetime",
    "pharmacy_start_datetime",
    "pharmacy_end_datetime"
]

for col in timestamp_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

print("Dataset prepared.")
print("Rows:", len(df_clean))

Dataset prepared.
Rows: 62422


In [6]:
service_pairs = {
    "Prescription": (
        "prescription_start_datetime",
        "prescription_end_datetime"
    ),
    "Lab": (
        "lab_start_datetime",
        "lab_end_datetime"
    ),
    "Imaging": (
        "imaging_start_datetime",
        "imaging_end_datetime"
    ),
    "Pharmacy": (
        "pharmacy_start_datetime",
        "pharmacy_end_datetime"
    )
}

In [7]:
duration_results = []
duration_quality = []

for activity, (start_col, end_col) in service_pairs.items():

    # Keep rows where both timestamps exist
    valid = df_clean[[start_col, end_col]].dropna()

    # Calculate duration in minutes
    durations = (
        valid[end_col] - valid[start_col]
    ).dt.total_seconds() / 60

    # Identify invalid and unusual durations
    negative_count = int((durations < 0).sum())
    zero_count = int((durations == 0).sum())
    over_24_hours = int((durations > 1440).sum())
    over_7_days = int((durations > 10080).sum())

    # Exclude negative durations from summary statistics
    valid_durations = durations[durations >= 0]

    # Store aggregate duration statistics
    duration_results.append({
        "activity": activity,
        "records_with_both_timestamps": len(valid),
        "negative_duration_records": negative_count,
        "valid_duration_records": len(valid_durations),
        "average_duration_minutes": (
            round(valid_durations.mean(), 2)
            if not valid_durations.empty else None
        ),
        "median_duration_minutes": (
            round(valid_durations.median(), 2)
            if not valid_durations.empty else None
        )
    })

    # Store aggregate data-quality statistics
    duration_quality.append({
        "activity": activity,
        "records_with_both_timestamps": len(valid),
        "zero_duration_records": zero_count,
        "negative_duration_records": negative_count,
        "over_24_hours": over_24_hours,
        "over_7_days": over_7_days
    })

In [8]:
duration_summary = pd.DataFrame(duration_results)
duration_quality_summary = pd.DataFrame(duration_quality)

print("\nACTIVITY DURATION SUMMARY")
display(duration_summary)

print("\nDURATION DATA QUALITY SUMMARY")
display(duration_quality_summary)



ACTIVITY DURATION SUMMARY


,activity,records_with_both_timestamps,negative_duration_records,valid_duration_records,average_duration_minutes,median_duration_minutes
0,Prescription,62409,52,62357,10.91,0.0
1,Lab,13327,0,13327,468.74,125.0
2,Imaging,5513,0,5513,96.68,3.0
3,Pharmacy,302,0,302,5.36,1.0



DURATION DATA QUALITY SUMMARY


,activity,records_with_both_timestamps,zero_duration_records,negative_duration_records,over_24_hours,over_7_days
0,Prescription,62409,41514,52,65,21
1,Lab,13327,1,0,732,51
2,Imaging,5513,561,0,26,9
3,Pharmacy,302,143,0,0,0


In [9]:
duration_summary.to_csv(
    "careflow_activity_duration_summary.csv",
    index=False
)

duration_quality_summary.to_csv(
    "careflow_duration_quality_summary.csv",
    index=False
)

print("\nAnalysis completed successfully!")
print("Saved: careflow_activity_duration_summary.csv")
print("Saved: careflow_duration_quality_summary.csv")



Analysis completed successfully!
Saved: careflow_activity_duration_summary.csv
Saved: careflow_duration_quality_summary.csv
